
# PRCP-1002 — Handwritten Digits Recognition (MNIST)
Tasks 
1) Prepare a complete data analysis report on the given data.  
2) Classify a given image of a handwritten digit (0–9).  
3) Compare multiple models and identify the best classifier for production.


## 0. Environment & Setup

If you see `ModuleNotFoundError: No module named 'tensorflow'`, run the cell below to install dependencies.

> **Note:** GPU is optional. If you have a CUDA-enabled GPU, ensure your drivers/CUDA are set up before installing TensorFlow.


In [ ]:
%pip install -q tensorflow scikit-learn pandas matplotlib pillow
%pip install -q scikit-learn-intelex


In [ ]:

import os, math, itertools, time, json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.utils import to_categorical

# sklearn (classical ML)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)


In [ ]:

(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)
classes, counts = np.unique(y_train, return_counts=True)
print("Class distribution (train):", dict(zip(classes, counts)))



## 2. Exploratory Data Analysis (EDA)

We will:
- Preview a few samples per class
- Plot label distribution
- Inspect pixel intensity stats


In [ ]:

# Plot a grid of digits (one per class where possible)
plt.figure(figsize=(10, 4))
shown = set()
idx = 1
for i in range(len(x_train)):
    if y_train[i] not in shown:
        plt.subplot(2, 5, idx)
        plt.imshow(x_train[i], cmap='gray')
        plt.title(f"Label: {y_train[i]}")
        plt.axis('off')
        shown.add(y_train[i])
        idx += 1
        if idx > 10:
            break
plt.tight_layout()
plt.show()




In [ ]:
# Label histogram
plt.figure(figsize=(6,4))
plt.hist(y_train, bins=np.arange(-0.5,10.5,1), rwidth=0.8)
plt.title("Label Distribution (Train)")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

# Pixel stats
print("Pixel intensity range (train):", x_train.min(), "to", x_train.max())
print("Mean/std (train):", float(x_train.mean()), float(x_train.std()))


## 3. Preprocessing

- Normalize pixel values to `[0,1]`  
- Create a validation set from the original training data  
- Prepare two views of features:  
  - Flattened(for classical ML & MLP) → shape   
  - Image with channel (for CNN) → shape 


In [ ]:

# Normalize to [0, 1]
x_train_f = x_train.astype('float32') / 255.0
x_test_f  = x_test.astype('float32') / 255.0

# Train/Validation split
x_tr, x_val, y_tr, y_val = train_test_split(x_train_f, y_train, test_size=0.1, random_state=42, stratify=y_train)

# Flattened
x_tr_flat  = x_tr.reshape((x_tr.shape[0], -1))
x_val_flat = x_val.reshape((x_val.shape[0], -1))
x_test_flat= x_test_f.reshape((x_test_f.shape[0], -1))

# With channel for CNN
x_tr_img  = np.expand_dims(x_tr, -1)
x_val_img = np.expand_dims(x_val, -1)
x_test_img= np.expand_dims(x_test_f, -1)

print("x_tr_flat:", x_tr_flat.shape, "x_tr_img:", x_tr_img.shape)


We compare several quick baselines:
- Logistic Regression (saga)
- Linear SVM (LinearSVC) 
- RBF SVM (SVC, subsampled for speed)
- KNN (k=3)  
- Random Forest 
We evaluate accuracy and macro F1 on the validation set.

In [ ]:

results = []

def eval_and_store(name, model, X_val, y_val):
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1  = f1_score(y_val, y_pred, average='macro')
    results.append({"Model": name, "Val_Accuracy": acc, "Val_MacroF1": f1})
    print(f"{name}: acc={acc:.4f}, macroF1={f1:.4f}")
    return y_pred

# Standardize for linear models / KNN
scaler = StandardScaler()
x_tr_flat_std  = scaler.fit_transform(x_tr_flat)
x_val_flat_std = scaler.transform(x_val_flat)

# Logistic Regression
lr = LogisticRegression(max_iter=200, solver='saga', n_jobs=-1, verbose=0)
lr.fit(x_tr_flat_std, y_tr)
eval_and_store("LogReg (saga)", lr, x_val_flat_std, y_val)

# Linear SVM
lsvc = LinearSVC()
lsvc.fit(x_tr_flat_std, y_tr)
eval_and_store("LinearSVC", lsvc, x_val_flat_std, y_val)

# RBF SVM (subsample to speed up training if needed)
sub = min(20000, x_tr_flat_std.shape[0])
svc = SVC(kernel='rbf', gamma='scale')
svc.fit(x_tr_flat_std[:sub], y_tr[:sub])
eval_and_store("RBF SVM (subsampled)", svc, x_val_flat_std, y_val)

# KNN
knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
knn.fit(x_tr_flat_std, y_tr)
eval_and_store("KNN (k=3)", knn, x_val_flat_std, y_val)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(x_tr_flat, y_tr)  # tree models don't need scaling
eval_and_store("RandomForest (200)", rf, x_val_flat, y_val)

pd.DataFrame(results)



## 5. MLP (Dense Neural Network)

A simple MLP on flattened pixels:
- Dense(512, ReLU) → Dropout(0.3)
- Dense(256, ReLU) → Dropout(0.3)
- Dense(10, Softmax)


In [ ]:

mlp_model = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

mlp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_mlp = mlp_model.fit(x_tr_flat, y_tr, epochs=8, batch_size=256, validation_data=(x_val_flat, y_val), verbose=2)

val_loss, val_acc = mlp_model.evaluate(x_val_flat, y_val, verbose=0)
print("MLP Val Accuracy:", val_acc)



## 6. CNN (Convolutional Neural Network)

A compact CNN:
- Conv(32,3) → BN → ReLU → MaxPool(2)  
- Conv(64,3) → BN → ReLU → MaxPool(2)  
- Flatten → Dense(128, ReLU) → Dropout(0.4) → Dense(10, Softmax)


In [ ]:

cnn_model = Sequential([
    Conv2D(32, (3,3), padding='same', input_shape=(28,28,1)),
    BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), padding='same'),
    BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax')
])

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_cnn = cnn_model.fit(x_tr_img, y_tr, epochs=6, batch_size=256, validation_data=(x_val_img, y_val), verbose=2)

val_loss, val_acc = cnn_model.evaluate(x_val_img, y_val, verbose=0)
print("CNN Val Accuracy:", val_acc)



## 7. Compare Models

We gather validation metrics from classical ML, MLP, and CNN to pick a production candidate.


In [ ]:

# Add neural nets to the results table
results.append({"Model": "MLP (Dense NN)", "Val_Accuracy": float(mlp_model.evaluate(x_val_flat, y_val, verbose=0)[1]), "Val_MacroF1": float(f1_score(y_val, np.argmax(mlp_model.predict(x_val_flat, verbose=0), axis=1), average='macro'))})
results.append({"Model": "CNN (ConvNet)", "Val_Accuracy": float(cnn_model.evaluate(x_val_img, y_val, verbose=0)[1]), "Val_MacroF1": float(f1_score(y_val, np.argmax(cnn_model.predict(x_val_img, verbose=0), axis=1), average='macro'))})

df_results = pd.DataFrame(results).sort_values(by=["Val_Accuracy","Val_MacroF1"], ascending=False).reset_index(drop=True)
df_results



## 8. Final Evaluation on Test Set

Select the **best** model by validation accuracy. Evaluate on the held-out test set, show confusion matrix and classification report.


In [ ]:

# Pick best by Val_Accuracy
best_row = pd.DataFrame(results).sort_values(by="Val_Accuracy", ascending=False).iloc[0]
best_name = best_row["Model"]
print("Best model by validation accuracy:", best_name)

def predict_with_model(name):
    if name == "LogReg (saga)":
        y_pred = lr.predict(scaler.transform(x_test_flat))
    elif name == "LinearSVC":
        y_pred = lsvc.predict(scaler.transform(x_test_flat))
    elif name == "RBF SVM (subsampled)":
        y_pred = svc.predict(scaler.transform(x_test_flat))
    elif name == "KNN (k=3)":
        y_pred = knn.predict(scaler.transform(x_test_flat))
    elif name == "RandomForest (200)":
        y_pred = rf.predict(x_test_flat)
    elif name == "MLP (Dense NN)":
        y_pred = np.argmax(mlp_model.predict(x_test_flat, verbose=0), axis=1)
    elif name == "CNN (ConvNet)":
        y_pred = np.argmax(cnn_model.predict(x_test_img, verbose=0), axis=1)
    else:
        raise ValueError("Unknown model")
    return y_pred

y_pred_test = predict_with_model(best_name)
acc_test = accuracy_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test, average='macro')
print(f"Test Accuracy: {acc_test:.4f} | Test Macro F1: {f1_test:.4f}")




In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(6,6))
plt.imshow(cm, cmap='Blues')
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_test))


## 9. Single-Image Inference (Task 2)

Use the helper below to classify a local image file   
The function tries to auto-handle common issues (RGB → grayscale, resizing to 28×28, optional inversion).


In [ ]:
def load_and_prepare_image(path, invert_if_needed=True):
    img = Image.open(path).convert('L')           # grayscale
    img = img.resize((28,28))                     # resize
    arr = np.array(img).astype('float32')

    # Heuristic: if background is white and digit is dark, keep as-is;
    # if background is dark and digit is bright, invert.
    if invert_if_needed:
        if arr.mean() > 127:  # bright background, keep
            pass
        else:                 # dark background, invert
            arr = 255 - arr

    arr = arr / 255.0
    return arr

In [ ]:
def classify_image(path, model_name=None):
    if model_name is None:
        model_name = best_name
    arr = load_and_prepare_image(path)
    if model_name in ["MLP (Dense NN)", "LogReg (saga)", "LinearSVC", "RBF SVM (subsampled)", "KNN (k=3)"]:
        feat = arr.reshape(1, -1)
        if model_name == "MLP (Dense NN)":
            probs = mlp_model.predict(feat, verbose=0)[0]
            pred  = int(np.argmax(probs))
        else:
            # scale if needed
            if model_name in ["LogReg (saga)", "LinearSVC", "RBF SVM (subsampled)", "KNN (k=3)"]:
                feat = scaler.transform(feat)
            pred = int(eval(model_name.split(' (')[0].lower()).predict(feat)[0]) if False else None
            # Safer explicit mapping:
            if model_name == "LogReg (saga)":
                pred = int(lr.predict(feat)[0])
            elif model_name == "LinearSVC":
                pred = int(lsvc.predict(feat)[0])
            elif model_name == "RBF SVM (subsampled)":
                pred = int(svc.predict(feat)[0])
            elif model_name == "KNN (k=3)":
                pred = int(knn.predict(feat)[0])
    elif model_name in ["RandomForest (200)"]:
        feat = arr.reshape(1, -1)
        pred = int(rf.predict(feat)[0])
    elif model_name == "CNN (ConvNet)":
        feat = arr.reshape(1,28,28,1)
        probs = cnn_model.predict(feat, verbose=0)[0]
        pred  = int(np.argmax(probs))
    else:
        raise ValueError("Unknown model name")
    return pred

In [ ]:
if best_name == "CNN (ConvNet)":
    save_dir = "models"
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, "cnn_mnist.keras")
    cnn_model.save(path)
    print("Saved:", path)



## 11. Challenges & Mitigations (Report)

Data-related
- Class balance: MNIST is fairly balanced across digits; still monitored distribution & macro F1 to ensure fairness.
- Pixel intensity scale: Normalization to `[0,1]` stabilizes optimization and benefits SVM/MLP/CNN.
- Noise & writing styles: Simple CNN provides translation/shape invariance compared to linear models.

Modeling
- Overfitting risk: Used validation split, dropout, and batch normalization for the CNN; monitored val metrics.
- Compute/time constraints: Subsampled for RBF SVM; used smaller epochs with early-stopping-ready configs.
- Hyperparameters: Kept defaults reasonable for a first pass; can further tune (C/gamma for SVM, trees for RF, depth/filters for CNN).

Production Readiness
- Latency: CNN runs fast at inference (single forward pass).  
- Portability: Saved best model; added a robust single-image preprocessing helper.  
- Monitoring: Use confusion matrix to identify digits with higher error rates (often 4/9/5 confusions).

Conclusion: CNN typically gives the highest validation/test accuracy on MNIST and is the recommended production model, with MLP as a simpler fallback and Logistic Regression as a quick baseline.
